In [1]:
import pandas as pd
import geopandas as gpd

In [8]:
dzielnice_gdf = gpd.read_file("data/dzielnice_lublin.gml", encoding="UTF-8")
lokale_gdf = gpd.read_file("data/lokale_lublin.gml", encoding="UTF-8")

In [4]:
dzielnice_gdf.head()

,gml_id,nazwa,info,geometry
0,dzielnice_granice.1164,Wieniawa,None,"POLYGON ((8398914.54 5680522.87, 8398914.54 56..."
1,dzielnice_granice.1168,Sławinek,None,"POLYGON ((8397243.25 5680981.27, 8397304.275 5..."
2,dzielnice_granice.1174,Kalinowszczyzna,None,"POLYGON ((8401271.162 5680090.935, 8401272.63 ..."
3,dzielnice_granice.1177,Węglin Pd.,None,"POLYGON ((8395320.29 5678351.25, 8395285.53 56..."
4,dzielnice_granice.1186,Bronowice,None,"POLYGON ((8403064.64 5678903.88, 8403222.09 56..."


In [5]:
lokale_gdf.head()

,gml_id,serwis_rcn,teryt,tran_przestrzen_nazw,tran_lokalny_id_iip,tran_wersja_id,tran_rodzaj_trans,tran_rodzaj_rynku,tran_sprzedajacy,tran_kupujacy,...,lok_nr_lokalu,lok_funkcja,lok_liczba_izb,lok_nr_kond,lok_pow_uzyt,lok_pow_przyn,lok_cena_brutto,lok_vat,lok_adres,geometry
0,lokale.7652240,None,0609,PL.PZGiK.9349.RCN,5A4905BC-D6A2-40D6-8353-6E94AF99B6AD,NaN,wolnyRynek,NaN,osobaPrawna,osobaFizyczna,...,1_LOK,mieszkalna,NaN,NaN,35.8,NaN,NaN,NaN,NaN,POINT (749233.476 386125.694)
1,lokale.7702785,None,0663,PL.PZGiK.4884.RCN,98781cfc-2bb7-4d85-9664-c969a4b2692f,2013-02-21T14:26:22,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,...,33_LOK,garaz,1,0,396,NaN,20000,NaN,MSC:Lublin;UL:Kryształowa;NR_PORZ:30,POINT (744316.367 378735.781)
2,lokale.7691559,None,0663,PL.PZGiK.4884.RCN,dd4ba0a2-1bf5-4246-913b-643e6abbecad,2013-01-11T11:11:48,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,...,1_LOK,mieszkalna,3,4,45.84,NaN,160000,NaN,MSC:Lublin;UL:Puchacza;NR_PORZ:8,POINT (750569.449 380204.84)
3,lokale.7682760,None,0663,PL.PZGiK.4884.RCN,33e4defe-db48-4760-8efd-ce127c4b8d75,2013-03-14T09:57:40,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,...,1_LOK,mieszkalna,4,1,69.6,4.4,257000,NaN,MSC:Lublin;UL:Leszetyckiego;NR_PORZ:6,POINT (747737.827 384676.722)
4,lokale.7693503,None,0663,PL.PZGiK.4884.RCN,d2afa76b-927a-4325-9f0f-74c3c9fbc15a,2013-07-05T14:01:57,sprzedazBezprzetargowa,pierwotny,osobaPrawna,osobaFizyczna,...,86_LOK,mieszkalna,3,4,49.62,10.79,225000,16666.67,MSC:Lublin;UL:Strzeszewskiego;NR_PORZ:17,POINT (749805.079 384621.036)


Zmiana crs

In [9]:
lokale_gdf.crs

<Projected CRS: EPSG:2180>
Name: ETRF2000-PL / CS92
Axis Info [cartesian]:
- x[north]: Northing (metre)
- y[east]: Easting (metre)
Area of Use:
- name: Poland - onshore and offshore.
- bounds: (14.14, 49.0, 24.15, 55.93)
Coordinate Operation:
- name: Poland CS92
- method: Transverse Mercator
Datum: ETRF2000 Poland
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [10]:
dzielnice_gdf.crs

<Projected CRS: EPSG:2179>
Name: ETRF2000-PL / CS2000/24
Axis Info [cartesian]:
- x[north]: Northing (metre)
- y[east]: Easting (metre)
Area of Use:
- name: Poland - east of 22°30'E.
- bounds: (22.5, 49.0, 24.15, 54.41)
Coordinate Operation:
- name: Poland CS2000 zone 8
- method: Transverse Mercator
Datum: ETRF2000 Poland
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [11]:
dzielnice_gdf = dzielnice_gdf.to_crs("EPSG:2180")

In [12]:
dzielnice_gdf.crs

<Projected CRS: EPSG:2180>
Name: ETRF2000-PL / CS92
Axis Info [cartesian]:
- x[north]: Northing (metre)
- y[east]: Easting (metre)
Area of Use:
- name: Poland - onshore and offshore.
- bounds: (14.14, 49.0, 24.15, 55.93)
Coordinate Operation:
- name: Poland CS92
- method: Transverse Mercator
Datum: ETRF2000 Poland
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [15]:
# Funkcja do dodania kolumny dzielnica
def dodaj_dzielnice_do_lokali(gdf_lokale, gdf_dzielnice, nazwa_kolumny_dzielnica='dzielnica', nazwa_kolumny_nazwa='name'):


    gdf_result = gpd.sjoin(gdf_lokale, gdf_dzielnice[['geometry', nazwa_kolumny_nazwa]], 
                            how='left', predicate='within')
    
    if nazwa_kolumny_nazwa in gdf_result.columns:
        gdf_result[nazwa_kolumny_dzielnica] = gdf_result[nazwa_kolumny_nazwa]
        gdf_result = gdf_result.drop(columns=[nazwa_kolumny_nazwa])
    
    if 'index_right' in gdf_result.columns:
        gdf_result = gdf_result.drop(columns=['index_right'])

    lokale_z_dzielnica = gdf_result[gdf_result[nazwa_kolumny_dzielnica].notna()].shape[0]
    lokale_bez_dzielnica = gdf_result[gdf_result[nazwa_kolumny_dzielnica].isna()].shape[0]
    
    print(f"  Lokale z przypisaną dzielnicą: {lokale_z_dzielnica}")
    print(f"  Lokale bez przypisanej dzielnicy: {lokale_bez_dzielnica}")
    
    
    return gdf_result


In [16]:
combined_gdf = dodaj_dzielnice_do_lokali(lokale_gdf, dzielnice_gdf, "dzielnica", "nazwa")

  Lokale z przypisaną dzielnicą: 96646
  Lokale bez przypisanej dzielnicy: 20


In [ ]:
combined_gdf.to_file("data/dane_epsg2180_lublin.gml", driver='GML')